# 08. Portfolio Construction

El objetivo de este notebook será transformar las predicciones OOS de Ridge, XGBoost y Random Forest en señales de inversión y posteriormente en carteras invertibles, evaluando cómo las distintas reglas de selección, asignación de pesos, restricciones y rebalanceo afectan al resultado económico.

El objetivo no es todavía realizar un análisis exhaustivo de rentabilidad y riesgo, sino construir de forma sistemática las carteras que posteriormente serán evaluadas en el Notebook 09.

## 1. Imports & Configuration

### 1.1 Librerías

In [1]:
import sys
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import joblib

from pathlib import Path
from scipy.stats import spearmanr

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

### 1.2 Configuración del notebook

El horizonte de predicción utilizado por los modelos es de 21 sesiones de trading, correspondiente al target forward_return_21d. Este horizonte define el periodo sobre el que se evalúa la capacidad predictiva de las señales, pero no determina automáticamente la frecuencia de rebalanceo de las carteras. Como configuración inicial se utilizará un rebalanceo cada 21 sesiones, que posteriormente se comparará con frecuencias alternativas para analizar su impacto sobre turnover, costes y estabilidad de las posiciones.

Las carteras benchmark se construirán a partir de los extremos del ranking cross-sectional, utilizando el 10% superior para posiciones long y, en estrategias long-short, el 10% inferior para posiciones short. Las exposiciones objetivo se mantendrán constantes entre modelos para garantizar una comparación homogénea, mientras que el número exacto de posiciones dependerá del universo disponible en cada fecha de rebalanceo.


In [2]:
# =============================================================================
# Experiment Configuration
# =============================================================================

# Prediction horizon
PREDICTION_HORIZON = 21

# Rebalancing
# Benchmark frequency; alternative frequencies will be evaluated later.
REBALANCING_FREQUENCY = 21

# Portfolio selection
SELECTION_PERCENTS = [0.10, 0.20, 0.30]

# Benchmark selection
BENCHMARK_SELECTION = 0.10

# Long-only exposure
LONG_ONLY_GROSS_EXPOSURE = 1.00

# Long-short exposure
LONG_SHORT_LONG_EXPOSURE = 0.50
LONG_SHORT_SHORT_EXPOSURE = 0.50

# Position selection
POSITION_COUNT_MODE = "quantile"

In [3]:
# =============================================================================
# Exposure Validation
# =============================================================================

assert (
    LONG_SHORT_LONG_EXPOSURE >= 0
    and LONG_SHORT_SHORT_EXPOSURE >= 0
)

assert (
    LONG_SHORT_LONG_EXPOSURE
    + LONG_SHORT_SHORT_EXPOSURE
    > 0
)

### 1.3 Rutas y parámetros globales

In [ ]:
# =============================================================================
# Paths
# =============================================================================

OOS_DATA_PATH = "../data/model_results/final_model/oos/"
PREDICTIONS_PATH = (OOS_DATA_PATH + "oos_predictions.parquet")

# Path to price data (OOS included)
PRICES_PATH = "../data/raw/sp500_prices_extended.parquet"

# =============================================================================
# Models
# =============================================================================

MODEL_COLUMNS = {
    "Ridge": "prediction_ridge_rank",
    "XGBoost": "prediction_xgb_rank",
    "Random Forest": "prediction_rf_rank",
}

# =============================================================================
# Required Columns
# =============================================================================

REQUIRED_COLUMNS = [
    "prediction_ridge_rank",
    "prediction_xgb_rank",
    "prediction_rf_rank",
    "forward_return_21d",
]

# =============================================================================
# Signal Configuration
# =============================================================================

PREDICTION_COLUMNS = list(MODEL_COLUMNS.values())

RANKING_METHOD = "cross_sectional_percentile"

## 2. Data Loading

### 2.1 Carga de predicciones OOS

In [5]:
# =============================================================================
# Load OOS Predictions
# =============================================================================

oos_predictions = pd.read_parquet(PREDICTIONS_PATH)

print(f"OOS predictions shape: {oos_predictions.shape}")
print(f"Date range: {oos_predictions.index.get_level_values('date').min()} "
      f"→ {oos_predictions.index.get_level_values('date').max()}")

OOS predictions shape: (184408, 7)
Date range: 2025-01-15 00:00:00 → 2026-07-10 00:00:00


In [6]:
# =============================================================================
# Prediction Columns Check
# =============================================================================

missing_prediction_columns = [
    column
    for column in PREDICTION_COLUMNS
    if column not in oos_predictions.columns
]

assert not missing_prediction_columns, (
    f"Missing prediction columns: {missing_prediction_columns}"
)

# =============================================================================
# Index Check
# =============================================================================

assert isinstance(oos_predictions.index, pd.MultiIndex)
assert oos_predictions.index.names == ["date", "ticker"]

# =============================================================================
# Prediction Availability Check
# =============================================================================

assert oos_predictions[PREDICTION_COLUMNS].notna().any().all(), (
    "At least one prediction column contains no valid observations."
)

### 2.2 Carga de precios y retornos realizados

Se cargan los precios históricos necesarios para el backtest y se restringe el universo al conjunto de activos utilizado por las predicciones OOS. Se conserva toda la cobertura histórica disponible, ya que los precios anteriores al periodo OOS pueden ser necesarios para estimar medidas de riesgo en cada fecha de rebalanceo. A partir de los precios ajustados se calculan los retornos simples y logarítmicos, que se utilizarán posteriormente en la construcción y evaluación de las carteras.

In [7]:
# =============================================================================
# Load Extended Price Data
# =============================================================================

prices = pd.read_parquet(PRICES_PATH)

# =============================================================================
# Remove Securities Excluded During Development
# =============================================================================

tickers_to_remove = [
    "SW",
    "AMCR",
]

prices = prices.drop(
    columns=tickers_to_remove,
    level=1,
    errors="ignore",
)

# =============================================================================
# Align Price Universe with OOS Universe
# =============================================================================

oos_tickers = oos_predictions.index.get_level_values("ticker").unique()

price_tickers = prices.columns.get_level_values(1)

missing_price_tickers = oos_tickers.difference(price_tickers)

assert len(missing_price_tickers) == 0, (
    f"Missing price data for tickers: "
    f"{missing_price_tickers.tolist()}"
)

prices = prices.loc[
    :,
    prices.columns.get_level_values(1).isin(oos_tickers)
]

# =============================================================================
# Compute Returns
# =============================================================================

adj_close = prices["Adj Close"]

simple_returns = adj_close.pct_change()

log_returns = np.log(
    adj_close / adj_close.shift(1)
)

In [8]:
print(
    f"Price data coverage: "
    f"{prices.index.min().date()} → {prices.index.max().date()}"
)

print(f"Number of securities: {len(oos_tickers)}")

Price data coverage: 2010-01-04 → 2026-08-10
Number of securities: 496


### 2.3 Verificación de integridad y cobertura temporal

Antes de comenzar la construcción de las señales y carteras, se verifica que las predicciones OOS, los precios y los retornos presentan una cobertura temporal y un universo de activos compatibles. Se comprueba que todos los activos y fechas necesarios para el periodo OOS disponen de información de precios y que las series de retornos permanecen correctamente alineadas con los precios originales. Estas comprobaciones garantizan la consistencia de los datos utilizados en el backtest, mientras que la prevención de look-ahead bias se controlará posteriormente en cada etapa en la que se utilice información histórica para determinar señales, riesgo o pesos de cartera.

In [9]:
# =============================================================================
# Temporal Coverage
# =============================================================================

oos_dates = (
    oos_predictions.index
    .get_level_values("date")
    .unique()
)

assert oos_dates.min() >= prices.index.min()
assert oos_dates.max() <= prices.index.max()

# =============================================================================
# Ticker Coverage
# =============================================================================

oos_tickers = (
    oos_predictions.index
    .get_level_values("ticker")
    .unique()
)

price_tickers = (
    prices.columns
    .get_level_values(1)
    .unique()
)

missing_price_tickers = oos_tickers.difference(price_tickers)

assert len(missing_price_tickers) == 0, (
    f"Missing price data for tickers: "
    f"{missing_price_tickers.tolist()}"
)

# =============================================================================
# Prediction Date Coverage
# =============================================================================

missing_prediction_dates = oos_dates.difference(
    prices.index
)

assert len(missing_prediction_dates) == 0, (
    f"Missing price data for OOS dates: "
    f"{missing_prediction_dates.tolist()}"
)

# =============================================================================
# Returns Alignment
# =============================================================================

assert simple_returns.index.equals(prices.index)
assert log_returns.index.equals(prices.index)

assert simple_returns.columns.equals(adj_close.columns)
assert log_returns.columns.equals(adj_close.columns)

# =============================================================================
# Data Availability
# =============================================================================

assert simple_returns.notna().any().all(), (
    "Some securities contain no valid simple returns."
)

assert log_returns.notna().any().all(), (
    "Some securities contain no valid log returns."
)

## 3. Signal Construction

### 3.1 Cross-sectional Ranking

Las predicciones OOS de Ridge, XGBoost y Random Forest se transforman en rankings cross-sectional diarios para ordenar los activos según la intensidad relativa de su señal. El ranking se calcula únicamente sobre los activos que disponen de una predicción válida en cada fecha, respetando el universo efectivamente disponible durante el periodo OOS. Dado que los modelos basados en árboles presentan un número reducido de valores de predicción únicos, especialmente XGBoost, se utiliza un ranking determinista (method="first") para resolver los empates y garantizar una distribución aproximadamente equilibrada de los activos en los posteriores grupos de selección. Esta transformación no modifica las predicciones OOS originales.

In [ ]:
# =============================================================================
# Cross-sectional Ranking
# =============================================================================

signal_ranks = pd.DataFrame(
    index=oos_predictions.index
)

RANK_COLUMNS = {}

for model, prediction_column in MODEL_COLUMNS.items():

    rank_column = f"{model.lower().replace(' ', '_')}_rank"

    signal_ranks[rank_column] = (
        oos_predictions
        .groupby(level="date")[prediction_column]
        .rank(
            method="first",
            pct=True,
        )
    )

    RANK_COLUMNS[model] = rank_column

In [22]:
# =============================================================================
# Ranking Validation
# =============================================================================

for rank_column in RANK_COLUMNS.values():

    assert signal_ranks[rank_column].dropna().between(0, 1).all(), (
        f"Invalid values found in {rank_column}."
    )

# =============================================================================
# Cross-sectional Coverage
# =============================================================================

rank_counts = (
    signal_ranks
    .groupby(level="date")
    .size()
)

print(
    f"Cross-sectional observations per date: "
    f"Min = {rank_counts.min()} | "
    f"Max = {rank_counts.max()}"
)

Cross-sectional observations per date: Min = 494 | Max = 496


### 3.2 Quantile Formation

El ranking cross-sectional continuo se divide en diez grupos de tamaño aproximadamente equivalente, desde D1, que contiene los activos con las predicciones relativas más bajas, hasta D10, que contiene los activos con las predicciones más altas. La formación de los grupos se realiza diariamente, utilizando únicamente los activos disponibles en cada fecha.

Debido al elevado número de empates presente en las predicciones de los modelos basados en árboles, especialmente XGBoost, se utiliza un ranking determinista (method="first") para resolver los empates antes de formar los cuantiles. De esta forma se evitan grupos excesivamente desiguales y se garantiza una comparación homogénea entre modelos. Esta decisión únicamente afecta a la asignación de observaciones a grupos y no modifica las predicciones originales.


In [26]:
# =============================================================================
# Quantile Formation
# =============================================================================

quantile_data = pd.DataFrame(
    index=signal_ranks.index
)

QUANTILE_COLUMNS = {}

for model, rank_column in RANK_COLUMNS.items():

    quantile_column = (
        f"{model.lower().replace(' ', '_')}_quantile"
    )

    quantile_data[quantile_column] = (
        np.ceil(
            signal_ranks[rank_column] * 10
        )
        .clip(upper=10)
        .astype("Int64")
    )

    QUANTILE_COLUMNS[model] = quantile_column

In [27]:
# =============================================================================
# Quantile Distribution
# =============================================================================

quantile_counts = {}

for model, quantile_column in QUANTILE_COLUMNS.items():

    counts = (
        quantile_data
        .groupby(level="date")[quantile_column]
        .value_counts()
        .unstack(fill_value=0)
    )

    quantile_counts[model] = counts

for model, counts in quantile_counts.items():

    print("=" * 80)
    print(f"{model} — QUANTILE SIZE RANGE")
    print("=" * 80)

    print(
        f"Min = {counts.min().min()} | "
        f"Max = {counts.max().max()}"
    )

Ridge — QUANTILE SIZE RANGE
Min = 49 | Max = 50
XGBoost — QUANTILE SIZE RANGE
Min = 49 | Max = 50
Random Forest — QUANTILE SIZE RANGE
Min = 49 | Max = 50


### 3.3 Signal Distribution Analysis

Antes de utilizar los rankings para construir las carteras, se analiza su distribución cross-sectional durante el periodo OOS. El objetivo es comprobar que las señales presentan suficiente dispersión entre activos y detectar posibles concentraciones, valores extremos o diferencias estructurales entre modelos que puedan afectar a la selección y posterior asignación de pesos. Dado que los rankings se construyen mediante percentiles cross-sectional, se presta especial atención a la distribución de las predicciones originales y al comportamiento de los grupos por cuantiles.


#### 3.3.1 Distribución de las predicciones

In [29]:
# =============================================================================
# Prediction Distribution
# =============================================================================

prediction_distribution = (
    oos_predictions[
        list(MODEL_COLUMNS.values())
    ]
    .describe()
    .T[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

display(prediction_distribution)

# =============================================================================
# Unique Prediction Values
# =============================================================================

unique_predictions = pd.DataFrame(
    {
        model: [
            oos_predictions[prediction_column]
            .nunique()
        ]
        for model, prediction_column in MODEL_COLUMNS.items()
    },
    index=["Unique values"],
).T

display(unique_predictions)

,count,mean,std,min,25%,50%,75%,max
prediction_ridge_rank,184408.0,0.014807,0.004190,0.005012,0.011615,0.014853,0.017929,0.024854
prediction_xgb_rank,184408.0,0.014821,0.003660,0.013136,0.013136,0.013318,0.015174,0.057154
prediction_rf_rank,184408.0,0.015472,0.009112,0.011264,0.012054,0.013353,0.015931,0.159783


,Unique values
Ridge,182612
XGBoost,285
Random Forest,4680


El análisis descriptivo de las predicciones fuera de muestra revela marcadas diferencias en la naturaleza estadística de cada arquitectura. Mientras que el modelo lineal Ridge genera una distribución continua con una elevada granulatividad (182,612 valores únicos), los modelos basados en árboles muestran una clara concentración discreta. 

En particular, XGBoost comprime el universo de predicciones en apenas 285 valores únicos, provocando que más del $50\%$ de las observaciones colapsen en un valor idéntico ($0.013136$).Por su parte, Random Forest presenta una mayor variabilidad ($\sigma = 0.009112$) y una pronunciada asimetría hacia la cola derecha, alcanzando predicciones puntuales de hasta el $15.97\%$. 

A pesar de estas diferencias morfológicas en la distribución de las salidas, los tres modelos mantienen una media de retorno predicho altamente coherente entre sí ($\approx 1.48\% - 1.54\%$). Esta acusada presencia de empates en XGBoost y Random Forest ratifica la necesidad de aplicar un ranking determinista (method="first") para garantizar una división equitativa y homogénea del universo en cuantiles.

#### 3.3.2 Distribución de los rankings

In [30]:
# =============================================================================
# Ranking Distribution
# =============================================================================

rank_distribution = (
    signal_ranks
    .describe()
    .T[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

display(rank_distribution)

,count,mean,std,min,25%,50%,75%,max
ridge_rank,184408.0,0.501009,0.288675,0.002016,0.251012,0.50101,0.751012,1.0
xgboost_rank,184408.0,0.501009,0.288675,0.002016,0.251012,0.50101,0.751012,1.0
random_forest_rank,184408.0,0.501009,0.288675,0.002016,0.251012,0.50101,0.751012,1.0


Tras la transformación a percentiles diarios ($[0, 1]$), la distribución de las tres señales converge exactamente a una distribución uniforme teórica $\mathcal{U}(0, 1)$.Los tres modelos presentan una media y mediana centradas en $0.5010$ y una desviación estándar idéntica a la teórica ($\sigma \approx 0.2887$). 

Esta estandarización elimina las diferencias de escala previas entre algoritmos, garantizando un punto de partida homogéneo y simétrico para la división en deciles ($D1$ a $D10$).

#### 3.3.3 Distribución de los cuantiles

In [32]:
# =============================================================================
# Quantile Distribution
# =============================================================================

quantile_distribution = {}

for model, quantile_column in QUANTILE_COLUMNS.items():

    counts = (
        quantile_data[quantile_column]
        .value_counts()
        .sort_index()
    )

    quantile_distribution[model] = counts

quantile_distribution = pd.DataFrame(
    quantile_distribution
)

quantile_distribution.index.name = "Quantile"

display(quantile_distribution)

# =============================================================================
# Quantile Imbalance
# =============================================================================

quantile_imbalance = {}

for model, quantile_column in QUANTILE_COLUMNS.items():

    counts = (
        quantile_data
        .groupby(level="date")[quantile_column]
        .value_counts()
        .unstack(fill_value=0)
    )

    quantile_imbalance[model] = {
        "Minimum group size": counts.min().min(),
        "Maximum group size": counts.max().max(),
        "Maximum imbalance": (
            counts.max().max()
            - counts.min().min()
        ),
    }

quantile_imbalance = pd.DataFrame(
    quantile_imbalance
).T

display(quantile_imbalance)

,Ridge,XGBoost,Random Forest
Quantile,,,
1,18228,18228,18228
2,18550,18550,18550
3,18278,18278,18278
4,18550,18550,18550
5,18596,18596,18596
6,18232,18232,18232
7,18546,18546,18546
8,18282,18282,18282
9,18546,18546,18546


,Minimum group size,Maximum group size,Maximum imbalance
Ridge,49,50,1
XGBoost,49,50,1
Random Forest,49,50,1



## 4. Benchmark Portfolio Formation

Antes de introducir metodologías de asignación de pesos basadas en la intensidad de la señal, el riesgo o la optimización, se construyen carteras benchmark mediante reglas simples y transparentes. Estas carteras permiten establecer una referencia común frente a la que evaluar posteriormente si los métodos de weighting más sofisticados aportan mejoras en términos de rentabilidad, riesgo, concentración o costes de implementación.

Se utilizarán dos benchmarks principales. El primero será una estrategia long-only basada en el decil superior del ranking, que servirá como referencia para evaluar diferentes metodologías de asignación dentro de una cartera con exposición positiva. El segundo será una estrategia long-short que combina el decil superior e inferior, permitiendo evaluar la capacidad de las señales para diferenciar entre activos con expectativas relativas altas y bajas.

En ambos casos se utilizará Equal Weight como regla de asignación inicial. De esta forma, la selección de activos y la asignación de pesos quedan separadas: los rankings determinan qué activos entran en la cartera, mientras que el benchmark asigna el mismo peso a todas las posiciones seleccionadas. Estas carteras constituirán la referencia frente a la que se compararán posteriormente las metodologías de Signal Weighting, Risk-Based Allocation y Mathematical Optimization.


In [ ]:
# =============================================================================
# Portfolio Weights Structure
# =============================================================================

portfolio_weights = pd.DataFrame(
    columns=[
        "date",
        "ticker",
        "model",
        "portfolio",
        "weight",
    ]
)


### 4.1 Long-only Top 10% Equal Weight

Se construye una cartera long-only utilizando el decil superior del ranking de cada modelo como universo de inversión. Todos los activos seleccionados reciben el mismo peso, de forma que la cartera mantiene una exposición bruta y neta de 100% sin introducir información adicional sobre la intensidad de la señal o el riesgo individual de los activos. Esta estrategia constituye el benchmark principal para evaluar posteriormente si metodologías de weighting más sofisticadas aportan valor adicional respecto a una asignación simple y robusta. La selección se realiza de forma independiente para cada modelo y fecha de rebalanceo, y los pesos se almacenan en una estructura común para facilitar su posterior comparación.


In [39]:
# =============================================================================
# Long-only Top 10% Equal Weight
# =============================================================================

long_only_weights_list = []

for model, quantile_column in QUANTILE_COLUMNS.items():

    # -------------------------------------------------------------------------
    # Select top decile
    # -------------------------------------------------------------------------

    selected = (
        quantile_data[quantile_column] == 10
    )

    selected_index = (
        quantile_data.index[selected]
    )

    # -------------------------------------------------------------------------
    # Count selected positions per date
    # -------------------------------------------------------------------------

    selected_counts = (
        pd.Series(
            1,
            index=selected_index,
        )
        .groupby(level="date")
        .sum()
    )

    # -------------------------------------------------------------------------
    # Equal weights
    # -------------------------------------------------------------------------

    weights = (
        pd.Series(
            1.0,
            index=selected_index,
        )
        .div(
            selected_counts,
            level="date",
        )
    )

    # -------------------------------------------------------------------------
    # Build model portfolio
    # -------------------------------------------------------------------------

    model_weights = (
        weights
        .rename("weight")
        .reset_index()
    )

    model_weights["model"] = model
    model_weights["portfolio"] = (
        "long_only_equal_weight"
    )

    long_only_weights_list.append(
        model_weights[
            [
                "date",
                "ticker",
                "model",
                "portfolio",
                "weight",
            ]
        ]
    )


# =============================================================================
# Combine models
# =============================================================================

long_only_weights = pd.concat(
    long_only_weights_list,
    ignore_index=True,
)

In [40]:
# =============================================================================
# Validation
# =============================================================================

print("=" * 80)
print("LONG-ONLY TOP 10% EQUAL WEIGHT — VALIDATION")
print("=" * 80)


# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    long_only_weights["weight"] >= 0
).all(), (
    "Negative weights detected."
)

print("✓ All portfolio weights are non-negative.")


# =============================================================================
# 2. One observation per ticker, date and model
# =============================================================================

duplicates = (
    long_only_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model observations."
)

print(
    "✓ No duplicated date-ticker-model observations."
)


# =============================================================================
# 3. Portfolio exposure
# =============================================================================

long_only_exposure = (
    long_only_weights
    .groupby(
        [
            "date",
            "model",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    long_only_exposure.values,
    LONG_ONLY_GROSS_EXPOSURE,
), (
    "Portfolio exposure is not equal to "
    f"{LONG_ONLY_GROSS_EXPOSURE:.2f}."
)

print(
    f"✓ Portfolio exposure = "
    f"{LONG_ONLY_GROSS_EXPOSURE:.2f}."
)


# =============================================================================
# 4. Equal-weight validation
# =============================================================================

weight_check = (
    long_only_weights
    .groupby(
        [
            "date",
            "model",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert np.allclose(
    weight_check["min_weight"],
    weight_check["max_weight"],
), (
    "Weights are not equal within at least "
    "one portfolio."
)

print("✓ Equal weighting confirmed.")


# =============================================================================
# 5. Position count
# =============================================================================

position_count = (
    long_only_weights
    .groupby(
        [
            "date",
            "model",
        ]
    )["ticker"]
    .nunique()
)

print(
    "✓ Position count:"
)

print(
    f"  Min = {position_count.min()}"
)

print(
    f"  Max = {position_count.max()}"
)


# =============================================================================
# 6. Check expected Top 10% size
# =============================================================================

expected_position_count = (
    quantile_data
    .groupby(level="date")
    .size()
    * BENCHMARK_SELECTION
)

expected_position_count = (
    expected_position_count
    .round()
    .astype(int)
)

actual_position_count = (
    position_count
    .groupby(level="date")
    .first()
)

print(
    "✓ Expected Top 10% position count:"
)

print(
    f"  Min = {expected_position_count.min()}"
)

print(
    f"  Max = {expected_position_count.max()}"
)


# =============================================================================
# 7. Weight range
# =============================================================================

print(
    "✓ Weight range:"
)

print(
    f"  Min = {long_only_weights['weight'].min():.6f}"
)

print(
    f"  Max = {long_only_weights['weight'].max():.6f}"
)


# =============================================================================
# 8. Final summary
# =============================================================================

print("\n" + "=" * 80)
print("VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 80)

LONG-ONLY TOP 10% EQUAL WEIGHT — VALIDATION
✓ All portfolio weights are non-negative.
✓ No duplicated date-ticker-model observations.
✓ Portfolio exposure = 1.00.
✓ Equal weighting confirmed.
✓ Position count:
  Min = 50
  Max = 50
✓ Expected Top 10% position count:
  Min = 49
  Max = 50
✓ Weight range:
  Min = 0.020000
  Max = 0.020000

VALIDATION COMPLETED SUCCESSFULLY


### 4.2 Long-short D10/D1 Equal Weight

Se construye una cartera long-short combinando el decil superior (D10) y el decil inferior (D1) del ranking de cada modelo. Los activos de D10 reciben posiciones largas con una exposición agregada del 50%, mientras que los activos de D1 reciben posiciones cortas con una exposición agregada del 50%. Dentro de cada lado se utiliza Equal Weight, de forma que todos los activos seleccionados reciben la misma magnitud de peso dentro de su respectivo lado.

Esta configuración genera una cartera market-neutral en términos de exposición, con una exposición neta objetivo de 0% y una exposición bruta de 100%. De este modo, el rendimiento de la estrategia depende principalmente de la capacidad del modelo para diferenciar entre los activos situados en los extremos superior e inferior del ranking, en lugar de depender de una exposición direccional al mercado.

La elección de una exposición del 50% por lado se mantiene constante para los tres modelos y constituye el benchmark frente al que posteriormente se evaluarán otras metodologías de weighting.

In [41]:
# =============================================================================
# 4.2 Long-short D10/D1 Equal Weight
# =============================================================================

long_short_weights_list = []

TARGET_LONG_EXPOSURE = 0.50
TARGET_SHORT_EXPOSURE = -0.50

for model, quantile_column in QUANTILE_COLUMNS.items():

    # -------------------------------------------------------------------------
    # Select D10 (Longs) and D1 (Shorts)
    # -------------------------------------------------------------------------
    selected_long = quantile_data[quantile_column] == 10
    selected_short = quantile_data[quantile_column] == 1

    index_long = quantile_data.index[selected_long]
    index_short = quantile_data.index[selected_short]

    # -------------------------------------------------------------------------
    # Count selected positions per date
    # -------------------------------------------------------------------------
    count_long = (
        pd.Series(1, index=index_long)
        .groupby(level="date")
        .sum()
    )

    count_short = (
        pd.Series(1, index=index_short)
        .groupby(level="date")
        .sum()
    )

    # -------------------------------------------------------------------------
    # Calculate equal weights per side (+0.50 / N_long and -0.50 / N_short)
    # -------------------------------------------------------------------------
    weights_long = (
        pd.Series(TARGET_LONG_EXPOSURE, index=index_long)
        .div(count_long, level="date")
    )

    weights_short = (
        pd.Series(TARGET_SHORT_EXPOSURE, index=index_short)
        .div(count_short, level="date")
    )

    # -------------------------------------------------------------------------
    # Combine sides and build model portfolio
    # -------------------------------------------------------------------------
    combined_weights = pd.concat([weights_long, weights_short]).sort_index()

    model_weights = (
        combined_weights
        .rename("weight")
        .reset_index()
    )

    model_weights["model"] = model
    model_weights["portfolio"] = "long_short_equal_weight"

    long_short_weights_list.append(
        model_weights[
            [
                "date",
                "ticker",
                "model",
                "portfolio",
                "weight",
            ]
        ]
    )

# =============================================================================
# Combine models
# =============================================================================

long_short_weights = pd.concat(
    long_short_weights_list,
    ignore_index=True,
)

In [42]:
# =============================================================================
# Validation
# =============================================================================

print("=" * 80)
print("LONG-SHORT D10/D1 EQUAL WEIGHT — VALIDATION")
print("=" * 80)


# =============================================================================
# 1. No duplicated observations
# =============================================================================

duplicates = (
    long_short_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated date-ticker-model observations."
)

print("✓ No duplicated date-ticker-model observations.")


# =============================================================================
# 2. Side exposure verification (Long = +0.50, Short = -0.50)
# =============================================================================

long_exposure = (
    long_short_weights[long_short_weights["weight"] > 0]
    .groupby(["date", "model"])["weight"]
    .sum()
)

short_exposure = (
    long_short_weights[long_short_weights["weight"] < 0]
    .groupby(["date", "model"])["weight"]
    .sum()
)

assert np.allclose(long_exposure.values, 0.50), (
    "Long side exposure is not equal to +0.50."
)

assert np.allclose(short_exposure.values, -0.50), (
    "Short side exposure is not equal to -0.50."
)

print("✓ Side exposures confirmed: Long = +0.50 | Short = -0.50.")


# =============================================================================
# 3. Gross and Net exposure verification (Gross = 1.00, Net = 0.00)
# =============================================================================

gross_exposure = (
    long_short_weights
    .assign(abs_weight=long_short_weights["weight"].abs())
    .groupby(["date", "model"])["abs_weight"]
    .sum()
)

net_exposure = (
    long_short_weights
    .groupby(["date", "model"])["weight"]
    .sum()
)

assert np.allclose(gross_exposure.values, 1.00), (
    "Gross exposure is not equal to 1.00."
)

assert np.allclose(net_exposure.values, 0.00, atol=1e-8), (
    "Net exposure is not equal to 0.00."
)

print("✓ Portfolio exposure metrics confirmed: Gross = 1.00 | Net = 0.00.")


# =============================================================================
# 4. Equal weighting per side
# =============================================================================

long_check = (
    long_short_weights[long_short_weights["weight"] > 0]
    .groupby(["date", "model"])["weight"]
    .agg(min_w="min", max_w="max")
)

short_check = (
    long_short_weights[long_short_weights["weight"] < 0]
    .groupby(["date", "model"])["weight"]
    .agg(min_w="min", max_w="max")
)

assert np.allclose(long_check["min_w"], long_check["max_w"]), (
    "Weights on the Long side are not equal."
)

assert np.allclose(short_check["min_w"], short_check["max_w"]), (
    "Weights on the Short side are not equal."
)

print("✓ Equal weighting per side confirmed.")


# =============================================================================
# 5. Position count per side (Longs vs Shorts)
# =============================================================================

long_counts = (
    long_short_weights[long_short_weights["weight"] > 0]
    .groupby(["date", "model"])["ticker"]
    .nunique()
)

short_counts = (
    long_short_weights[long_short_weights["weight"] < 0]
    .groupby(["date", "model"])["ticker"]
    .nunique()
)

print("✓ Position counts:")
print(f"  Long side  (D10): Min = {long_counts.min()} | Max = {long_counts.max()}")
print(f"  Short side  (D1): Min = {short_counts.min()} | Max = {short_counts.max()}")


# =============================================================================
# 6. Weight magnitude range
# =============================================================================

min_long_weight = long_short_weights[long_short_weights["weight"] > 0]["weight"].min()
max_long_weight = long_short_weights[long_short_weights["weight"] > 0]["weight"].max()

min_short_weight = long_short_weights[long_short_weights["weight"] < 0]["weight"].min()
max_short_weight = long_short_weights[long_short_weights["weight"] < 0]["weight"].max()

print("✓ Weight magnitude ranges:")
print(f"  Long weights  : [{min_long_weight:.6f}, {max_long_weight:.6f}]")
print(f"  Short weights : [{min_short_weight:.6f}, {max_short_weight:.6f}]")


# =============================================================================
# 7. Final summary
# =============================================================================

print("\n" + "=" * 80)
print("LONG-SHORT VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 80)

LONG-SHORT D10/D1 EQUAL WEIGHT — VALIDATION
✓ No duplicated date-ticker-model observations.
✓ Side exposures confirmed: Long = +0.50 | Short = -0.50.
✓ Portfolio exposure metrics confirmed: Gross = 1.00 | Net = 0.00.
✓ Equal weighting per side confirmed.
✓ Position counts:
  Long side  (D10): Min = 50 | Max = 50
  Short side  (D1): Min = 49 | Max = 49
✓ Weight magnitude ranges:
  Long weights  : [0.010000, 0.010000]
  Short weights : [-0.010204, -0.010204]

LONG-SHORT VALIDATION COMPLETED SUCCESSFULLY



### 4.3 Portfolio Exposure

En este apartado se sintetizan y consolidan las métricas agregadas de exposición para las carteras de referencia construidas en las secciones anteriores (*Long-Only Top 10%* y *Long-Short D10/D1*). Para cualquier cartera $p$ en una fecha $t$, las exposiciones se definen de la siguiente forma:

- **Exposición Larga ($E_L$):** Suma de los pesos positivos, $E_{L,t} = \sum_{w_{i,t} > 0} w_{i,t}$
- **Exposición Corta ($E_S$):** Suma de los pesos negativos, $E_{S,t} = \sum_{w_{i,t} < 0} w_{i,t}$
- **Exposición Bruta ($E_{\text{gross}}$):** Suma de las magnitudes absolutas de los pesos, $E_{\text{gross},t} = \sum |w_{i,t}| = E_{L,t} + |E_{S,t}|$
- **Exposición Neta ($E_{\text{net}}$):** Suma algebraica de todos los pesos, $E_{\text{net},t} = \sum w_{i,t} = E_{L,t} + E_{S,t}$

A continuación se calcula el perfil de exposición diario medio para cada combinación de modelo y cartera con el fin de verificar formalmente el perfil de exposición de cada cartera y comprobar la neutralidad de exposición neta en la estrategia long-short, así como la invariancia de la exposición entre modelos.

In [ ]:
benchmark_portfolios = pd.concat(
    [long_only_weights, long_short_weights],
    ignore_index=True,
)

daily_exposure = (
    benchmark_portfolios
    .groupby(["portfolio", "model", "date"])
    .agg(
        long_exp=("weight", lambda x: x[x > 0].sum()),
        short_exp=("weight", lambda x: x[x < 0].sum()),
        gross_exp=("weight", lambda x: x.abs().sum()),
        net_exp=("weight", "sum"),
    )
)

exposure_summary = (
    daily_exposure
    .groupby(["portfolio", "model"])
    .mean()
    .round(2)
    .reset_index()
)

print("=" * 80)
print("BENCHMARK PORTFOLIOS — EXPOSURE SUMMARY")
print("=" * 80)
display(exposure_summary.style.hide(axis="index"))

# =============================================================================
# Validation
# =============================================================================

# Silent assertions for Long-Only
lo_mask = exposure_summary["portfolio"] == "long_only_equal_weight"
assert np.allclose(exposure_summary.loc[lo_mask, "gross_exp"], 1.00), "Error in Long-Only Gross Exposure"
assert np.allclose(exposure_summary.loc[lo_mask, "net_exp"], 1.00), "Error in Long-Only Net Exposure"
assert np.allclose(exposure_summary.loc[lo_mask, "short_exp"], 0.00), "Error in Long-Only Short Exposure"

# Silent assertions for Long-Short
ls_mask = exposure_summary["portfolio"] == "long_short_equal_weight"
assert np.allclose(exposure_summary.loc[ls_mask, "gross_exp"], 1.00), "Error in Long-Short Gross Exposure"
assert np.allclose(exposure_summary.loc[ls_mask, "net_exp"], 0.00, atol=1e-6), "Error in Long-Short Net Exposure"
assert np.allclose(exposure_summary.loc[ls_mask, "long_exp"], 0.50), "Error in Long-Short Long Exposure"
assert np.allclose(exposure_summary.loc[ls_mask, "short_exp"], -0.50), "Error in Long-Short Short Exposure"

print("✓ All exposure checks passed successfully.")

BENCHMARK PORTFOLIOS — EXPOSURE SUMMARY


portfolio,model,long_exp,short_exp,gross_exp,net_exp
long_only_equal_weight,Random Forest,1.000000,0.000000,1.000000,1.000000
long_only_equal_weight,Ridge,1.000000,0.000000,1.000000,1.000000
long_only_equal_weight,XGBoost,1.000000,0.000000,1.000000,1.000000
long_short_equal_weight,Random Forest,0.500000,-0.500000,1.000000,0.000000
long_short_equal_weight,Ridge,0.500000,-0.500000,1.000000,0.000000
long_short_equal_weight,XGBoost,0.500000,-0.500000,1.000000,0.000000


✓ All exposure checks passed successfully.


Las carteras de referencia quedan validadas con total precisión matemática en los tres modelos. La estrategia Long-Only mantiene una exposición neta constante del 100% en el decil superior ($D10$), mientras que la Long-Short logra una neutralidad de mercado perfecta ($E_{\text{net}} = 0.00$) con un 50% de exposición por lado ($D10$ vs $D1$). Sin sesgos de escala ni diferencias de exposición entre algoritmos, el marco queda listo para pasar a la Sección 5.

## 5. Portfolio Selection Sensitivity

### 5.1 Top 10% vs. Top 20% vs. Top 30%

Se analiza cómo cambia la composición de las carteras long-only al ampliar progresivamente el universo de activos seleccionados. Manteniendo el mismo esquema de Equal Weight, se comparan los percentiles superiores del 10%, 20% y 30% para evaluar el compromiso entre concentración en las señales más extremas y diversificación de la cartera.

In [52]:
# =============================================================================
# Top 10% vs. Top 20% vs. Top 30% — Equal Weight
# =============================================================================

SELECTION_LEVELS = {
    "top_10": 10,
    "top_20": 9,
    "top_30": 8,
}

selection_weights_list = []


for model, quantile_column in QUANTILE_COLUMNS.items():

    for portfolio_name, minimum_quantile in SELECTION_LEVELS.items():

        # ---------------------------------------------------------------------
        # Select top percentile
        # ---------------------------------------------------------------------

        selected = (
            quantile_data[quantile_column]
            >= minimum_quantile
        )

        selected_index = (
            quantile_data.index[selected]
        )

        # ---------------------------------------------------------------------
        # Count selected positions per date
        # ---------------------------------------------------------------------

        selected_counts = (
            pd.Series(
                1,
                index=selected_index,
            )
            .groupby(level="date")
            .sum()
        )

        # ---------------------------------------------------------------------
        # Equal weights
        # ---------------------------------------------------------------------

        weights = (
            pd.Series(
                1.0,
                index=selected_index,
            )
            .div(
                selected_counts,
                level="date",
            )
        )

        # ---------------------------------------------------------------------
        # Build model portfolio
        # ---------------------------------------------------------------------

        model_weights = (
            weights
            .rename("weight")
            .reset_index()
        )

        model_weights["model"] = model

        model_weights["portfolio"] = (
            f"long_only_{portfolio_name}_equal_weight"
        )

        selection_weights_list.append(
            model_weights[
                [
                    "date",
                    "ticker",
                    "model",
                    "portfolio",
                    "weight",
                ]
            ]
        )


# =============================================================================
# Combine models and selection levels
# =============================================================================

selection_weights = pd.concat(
    selection_weights_list,
    ignore_index=True,
)

In [57]:
# =============================================================================
# Validation
# =============================================================================

print("=" * 80)
print("LONG-ONLY SELECTION SENSITIVITY — VALIDATION")
print("=" * 80)


# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    selection_weights["weight"] >= 0
).all(), (
    "Negative weights detected."
)

print("✓ All portfolio weights are non-negative.")


# =============================================================================
# 2. No duplicated observations
# =============================================================================

duplicates = (
    selection_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
            "portfolio",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model-portfolio observations."
)

print(
    "✓ No duplicated "
    "date-ticker-model-portfolio observations."
)


# =============================================================================
# 3. Portfolio exposure
# =============================================================================

portfolio_exposure = (
    selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    portfolio_exposure.values,
    1.00,
), (
    "Portfolio exposure is not equal to 1.00."
)

print(
    "✓ Portfolio exposure = 1.00 "
    "for all portfolios."
)


# =============================================================================
# 4. Equal-weight validation
# =============================================================================

weight_check = (
    selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert np.allclose(
    weight_check["min_weight"],
    weight_check["max_weight"],
), (
    "Weights are not equal within at least "
    "one portfolio."
)

print(
    "✓ Equal weighting confirmed "
    "for all portfolios."
)


# =============================================================================
# 5. Position count
# =============================================================================

position_count = (
    selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["ticker"]
    .nunique()
)

print("✓ Position count:")

for portfolio in [
    "long_only_top_10_equal_weight",
    "long_only_top_20_equal_weight",
    "long_only_top_30_equal_weight",
]:

    counts = position_count[
        position_count.index
        .get_level_values("portfolio")
        == portfolio
    ]

    print(
        f"  {portfolio}: "
        f"Min = {counts.min()} | "
        f"Max = {counts.max()}"
    )


# =============================================================================
# 6. Expected position count from quantile construction
# =============================================================================

expected_counts_list = []

for model, quantile_column in QUANTILE_COLUMNS.items():

    # -------------------------------------------------------------------------
    # Count assets selected by each quantile level
    # -------------------------------------------------------------------------

    model_counts = (
        quantile_data
        .groupby(level="date")[quantile_column]
        .agg(
            top_10=lambda x: (x >= 10).sum(),
            top_20=lambda x: (x >= 9).sum(),
            top_30=lambda x: (x >= 8).sum(),
        )
    )

    model_counts["model"] = model

    expected_counts_list.append(
        model_counts.reset_index()
    )


expected_counts = pd.concat(
    expected_counts_list,
    ignore_index=True,
)


print("✓ Expected position count ranges:")

for level in ["top_10", "top_20", "top_30"]:

    print(
        f"  {level}: "
        f"Min = {expected_counts[level].min()} | "
        f"Max = {expected_counts[level].max()}"
    )


# =============================================================================
# 7. Position count consistency
# =============================================================================

actual_counts = (
    position_count
    .reset_index()
)

actual_counts["selection_level"] = (
    actual_counts["portfolio"]
    .str.extract(r"top_(10|20|30)")[0]
)


actual_counts["selection_level"] = (
    "top_" + actual_counts["selection_level"]
)


actual_counts = (
    actual_counts
    .pivot(
        index=["date", "model"],
        columns="selection_level",
        values="ticker",
    )
    .reset_index()
)


expected_counts = (
    expected_counts
    .set_index(["date", "model"])
)


for level in ["top_10", "top_20", "top_30"]:

    actual = (
        actual_counts
        .set_index(["date", "model"])[level]
    )

    expected = expected_counts[level]

    assert (
        actual.equals(expected)
    ), (
        f"Position counts for {level} "
        "are not consistent with the "
        "underlying quantile construction."
    )


print(
    "✓ Position counts are consistent with "
    "the underlying quantile construction."
)


# =============================================================================
# 8. Weight range
# =============================================================================

print("✓ Weight ranges:")

for portfolio in [
    "long_only_top_10_equal_weight",
    "long_only_top_20_equal_weight",
    "long_only_top_30_equal_weight",
]:

    weights = selection_weights.loc[
        selection_weights["portfolio"] == portfolio,
        "weight",
    ]

    print(
        f"  {portfolio}: "
        f"[{weights.min():.6f}, "
        f"{weights.max():.6f}]"
    )


# =============================================================================
# 9. Selection ordering
# =============================================================================

position_count_table = (
    position_count
    .unstack("portfolio")
)

assert (
    position_count_table[
        "long_only_top_10_equal_weight"
    ]
    <=
    position_count_table[
        "long_only_top_20_equal_weight"
    ]
).all()

assert (
    position_count_table[
        "long_only_top_20_equal_weight"
    ]
    <=
    position_count_table[
        "long_only_top_30_equal_weight"
    ]
).all()

print(
    "✓ Position counts increase consistently "
    "from Top 10% → Top 20% → Top 30%."
)


# =============================================================================
# 10. Final summary
# =============================================================================

print("\n" + "=" * 80)
print(
    "SELECTION SENSITIVITY "
    "VALIDATION COMPLETED SUCCESSFULLY"
)
print("=" * 80)

LONG-ONLY SELECTION SENSITIVITY — VALIDATION
✓ All portfolio weights are non-negative.
✓ No duplicated date-ticker-model-portfolio observations.
✓ Portfolio exposure = 1.00 for all portfolios.
✓ Equal weighting confirmed for all portfolios.
✓ Position count:
  long_only_top_10_equal_weight: Min = 50 | Max = 50
  long_only_top_20_equal_weight: Min = 99 | Max = 100
  long_only_top_30_equal_weight: Min = 149 | Max = 149
✓ Expected position count ranges:
  top_10: Min = 50 | Max = 50
  top_20: Min = 99 | Max = 100
  top_30: Min = 149 | Max = 149


AssertionError: Position counts for top_10 are not consistent with the underlying quantile construction.


### 5.2 Long-short Quantile Sensitivity

Se estudia la sensibilidad de las carteras long-short al tamaño de los extremos seleccionados. Para ello, se comparan estrategias que toman posiciones largas y cortas en los extremos del 10%, 20% y 30% de la distribución, manteniendo constante la exposición bruta y la neutralidad de la exposición neta.



### 5.3 Position Count

Se analiza el número de posiciones resultante de cada nivel de selección y su relación con la concentración de la cartera. Este análisis permite cuantificar cómo la ampliación del universo seleccionado modifica el número de activos y, bajo Equal Weight, el peso individual asignado a cada posición.



## 6. Portfolio Weighting
En esta sección se estudian distintas metodologías para transformar la selección de activos en pesos concretos de cartera, desde métodos simples hasta técnicas que incorporan información sobre el riesgo y la optimización matemática.

### 6.1 Heuristic Allocation
Métodos sencillos y robustos que asignan los pesos sin necesidad de estimar matrices de covarianzas ni resolver problemas de optimización.

#### 6.1.1 Equal Weight
Asignación del mismo peso a todos los activos seleccionados, utilizando esta metodología como benchmark principal por su simplicidad y robustez.

#### 6.1.2 Signal Weighting
Asignación de mayores pesos a los activos con señales más fuertes, con el objetivo de aprovechar la intensidad relativa de las predicciones del modelo.

### 6.2 Risk-Based Allocation
Métodos que utilizan información sobre el riesgo de los activos para determinar los pesos, manteniendo la señal del modelo principalmente como mecanismo de selección.

#### 6.2.1 Inverse Volatility
Asignación de pesos inversamente proporcionales a la volatilidad reciente de cada activo, reduciendo la exposición a activos con mayor riesgo individual.

#### 6.2.2 Risk Parity
Asignación de pesos considerando tanto la volatilidad individual como las correlaciones entre activos, buscando una contribución más equilibrada de las distintas posiciones al riesgo total de la cartera.

### 6.3 Mathematical Optimization
Métodos que combinan las expectativas de retorno proporcionadas por los modelos con la estructura de riesgo de los activos para determinar los pesos de cartera mediante un problema de optimización.

#### 6.3.1 Mean-Variance / Maximum Sharpe
Aplicación de optimización media-varianza para buscar una combinación de pesos que maximice el Sharpe Ratio estimado, incorporando restricciones para evitar soluciones excesivamente concentradas o inestables.

## 7. Portfolio Constraints
Introducimos restricciones de cartera para controlar la concentración, las exposiciones y los riesgos derivados de los distintos métodos de asignación de pesos.

### 7.1 Maximum Position Weight
Limitación del peso máximo que puede alcanzar individualmente cada activo para evitar una concentración excesiva.

### 7.2 Exposure Constraints
Control de las exposiciones long, short, neta y bruta para garantizar que las carteras mantienen el perfil de riesgo definido.

### 7.3 Concentration Control
Evaluación y control de la concentración de las carteras para evitar que los pesos se concentren excesivamente en un reducido número de activos.

### 7.4 Optimization Constraints
Aplicación de restricciones específicas a los métodos de optimización matemática para evitar soluciones extremas, inestables o poco realistas desde el punto de vista de implementación.

### 7.5 Constraint Validation
Verificación de que los pesos finales cumplen todas las restricciones definidas después de completar la construcción de cada cartera.

## 8. Rebalancing
Transformamos las carteras estáticas en estrategias dinámicas mediante la actualización periódica de las posiciones.

### 8.1 Rebalancing Frequency
Definición de la frecuencia de actualización de las carteras, sin asumir automáticamente que el horizonte de predicción de 21 días implica un rebalanceo cada 21 sesiones.

### 8.2 Position Updates & Buffer Rules
Actualización de las posiciones y pesos de acuerdo con las nuevas señales disponibles en cada fecha de rebalanceo, incorporando bandas de tolerancia o reglas de buffer para mitigar ejecuciones marginales innecesarias.

### 8.3 Portfolio Turnover
Cálculo del volumen de posiciones que debe modificarse en cada rebalanceo como medida de la intensidad operativa de la estrategia.

## 9. Transaction Costs & Net Returns
Incorporamos los costes derivados de la implementación de las estrategias para obtener retornos económicamente más realistas.

### 9.1 Transaction Cost & Market Impact Model
Definición de un modelo explícito de costes de transacción que contemple comisiones de ejecución y estimaciones de bid-ask spread e impacto en el mercado aplicables a las operaciones generadas.

### 9.2 Cost per Rebalance
Cálculo de los costes asociados a los cambios de posiciones en cada periodo.

### 9.3 Gross Portfolio Returns
Cálculo de los retornos de las carteras antes de considerar los costes de transacción.

### 9.4 Net Portfolio Returns
Descuento de los costes de transacción para obtener los retornos netos de cada estrategia.

## 10. Portfolio Comparison
Comparamos las diferentes combinaciones de modelo, selección y metodología de weighting bajo unas condiciones de construcción homogéneas.

### 10.1 Ridge Portfolios
Construcción y almacenamiento de las carteras generadas a partir de las predicciones de Ridge utilizando las distintas metodologías de asignación de pesos.

### 10.2 XGBoost Portfolios
Construcción y almacenamiento de las carteras generadas a partir de las predicciones de XGBoost utilizando las mismas reglas.

### 10.3 Random Forest Portfolios
Construcción y almacenamiento de las carteras generadas a partir de las predicciones de Random Forest utilizando las mismas reglas.

### 10.4 Signal Comparison
Comparación de los rankings y de las selecciones producidas por los distintos modelos para analizar cómo las diferencias en las señales afectan a la composición de las carteras.

### 10.5 Weighting Comparison
Comparación de cómo las diferentes metodologías de asignación transforman una misma señal en carteras con diferentes niveles de concentración, riesgo y exposición.

### 10.6 Turnover & Cost Comparison
Comparación del turnover y de los costes de transacción generados por cada combinación de modelo y metodología de construcción.

## 11. Export Results
Guardamos los resultados necesarios para realizar posteriormente el análisis financiero completo de las estrategias en el Notebook 09.

### 11.1 Portfolio Weights
Exportación de los pesos de las carteras en cada fecha de rebalanceo.

### 11.2 Portfolio Returns
Exportación de los retornos brutos y netos de las estrategias.

### 11.3 Turnover
Exportación del turnover generado por cada cartera y periodo.

### 11.4 Transaction Costs
Exportación de los costes de transacción aplicados a cada estrategia.

### 11.5 Portfolio Metadata
Registro de los parámetros, reglas de selección, metodología de weighting, restricciones y supuestos utilizados para construir cada cartera.

## 12. Conclusions
Resumen de las principales características de las carteras construidas y de las diferencias observadas entre modelos, reglas de selección y metodologías de asignación de pesos.